### Table required for calculate loan score
1. customers_new -> home_ownership, grade, high_credit_limit
2. loans -> monthly_installment, load_status, funded_amount
3. loan_repayment -> last_payment, total_payment_received
4. loan_defaulter_delinq_new -> delinq_2_yrs
5. loan_defaulter_public_rec_new -> public_rec, pub_bankruptcies, inq_last_6mnths

### Loan Criteria
1. Payment history (ph) -> 20%  (last_payment, total_payment_amount)
2. Defaulter history (dh) -> 45% (delinq_3yrs,pub_rec, pub_bankruptices,inq_last_6mnths)
3. Financial health data (fh) -> 35% (home_ownership, loan_status, funded_amount, grade, subgrade)

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists gold;
use schema gold;
select current_catalog(), current_schema();

In [0]:
#setting config value for rating points

rating_points = {
    "unacceptable": 0,
    "verybad": 100,
    "bad": 250,
    "good": 500,
    "verygood": 650,
    "excellent": 800
}


In [0]:
#setting up final grade 

grade_points = {
    "unacceptable": 750,
    "verybad": 1000,
    "bad": 1500,
    "good": 2000,
    "verygood": 2500
}

#Payment history (ph)

In [0]:
query = f"""
SELECT l.member_id,
-- Last payment points
CASE
    WHEN lr.last_payment_amount < (l.installment * 0.5) THEN {rating_points['verybad']}
    WHEN lr.last_payment_amount >= (l.installment * 0.5) AND lr.last_payment_amount < l.installment THEN {rating_points['bad']}
    WHEN lr.last_payment_amount = l.installment THEN {rating_points['good']}
    WHEN lr.last_payment_amount >= l.installment AND lr.last_payment_amount < (l.installment * 1.5) THEN {rating_points['verygood']}
    WHEN lr.last_payment_amount >= (l.installment * 1.5) THEN {rating_points['excellent']}
    ELSE {rating_points['unacceptable']}
END AS last_payment_points,

-- Total payment points
CASE 
    WHEN lr.total_payment >= (l.funded_amount * 0.5) THEN {rating_points['verygood']}
    WHEN lr.total_payment < (l.funded_amount * 0.5) AND lr.total_payment > 0 THEN {rating_points['good']}
    WHEN lr.total_payment = 0 OR lr.total_payment IS NULL THEN {rating_points['unacceptable']}
END AS total_payment_points

FROM silver_cleaned.loans l
JOIN silver_cleaned.loans_repayment lr ON l.loan_id = lr.loan_id
"""

ph_df = spark.sql(query)


In [0]:
ph_df.show()

In [0]:
ph_df.createOrReplaceTempView('ph_points')

#Defualter history (dh)

In [0]:
query = f"""
SELECT ph.*,
-- Delinq_points
CASE
    WHEN dd.delinq_2yrs = 0 THEN {rating_points['excellent']}
    WHEN dd.delinq_2yrs BETWEEN 1 AND 2 THEN {rating_points['bad']}
    WHEN dd.delinq_2yrs BETWEEN 3 AND 5 THEN {rating_points['verybad']}
    WHEN dd.delinq_2yrs > 5 OR dd.delinq_2yrs IS NULL THEN {rating_points['unacceptable']}
END AS delinq_points,

-- Public_records
CASE
    WHEN dp.public_rec = 0 THEN {rating_points['excellent']}
    WHEN dp.public_rec BETWEEN 1 AND 2 THEN {rating_points['bad']}
    WHEN dp.public_rec BETWEEN 3 AND 5 THEN {rating_points['verybad']}
    WHEN dp.public_rec > 5 OR dp.public_rec IS NULL THEN {rating_points['unacceptable']}
END AS public_rec_points,

-- Public_bankruptcies_records
CASE
    WHEN dp.public_bankruptcies = 0 THEN {rating_points['excellent']}
    WHEN dp.public_bankruptcies BETWEEN 1 AND 2 THEN {rating_points['bad']}
    WHEN dp.public_bankruptcies BETWEEN 3 AND 5 THEN {rating_points['verybad']}
    WHEN dp.public_bankruptcies > 5 OR dp.public_bankruptcies IS NULL THEN {rating_points['unacceptable']}
END AS public_bankrupticies_points,

-- Inquiry
CASE
    WHEN dp.inquiry_6_months = 0 THEN {rating_points['excellent']}
    WHEN dp.inquiry_6_months BETWEEN 1 AND 2 THEN {rating_points['bad']}
    WHEN dp.inquiry_6_months BETWEEN 3 AND 5 THEN {rating_points['verybad']}
    WHEN dp.inquiry_6_months > 5 OR dp.inquiry_6_months IS NULL THEN {rating_points['unacceptable']}
END AS inquiry_points

FROM silver_enriched.loans_defaulter_delinq dd
JOIN silver_enriched.loans_defaulter_publicrecord dp ON dd.member_id = dp.member_id
JOIN ph_points ph ON dd.member_id = ph.member_id
"""

ph_dh_df = spark.sql(query)

In [0]:
ph_dh_df.show()

In [0]:
ph_dh_df.createOrReplaceTempView('ph_dh_points')

#Financial health (fh)

In [0]:
query = f"""
SELECT p.*,
-- Loan status points
CASE
    WHEN LOWER(l.loan_status) LIKE '%fully paid%' THEN {rating_points['excellent']}
    WHEN LOWER(l.loan_status) LIKE '%current%' THEN {rating_points['good']}
    WHEN LOWER(l.loan_status) LIKE '%in grace period%' THEN {rating_points['bad']}
    WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' THEN {rating_points['verybad']}
    WHEN LOWER(l.loan_status) LIKE '%charged off%' THEN {rating_points['unacceptable']}
    ELSE {rating_points['unacceptable']}
END AS loan_status_points,

-- Home ownership points
CASE
    WHEN LOWER(c.home_ownership) LIKE '%own%' THEN {rating_points['excellent']}
    WHEN LOWER(c.home_ownership) LIKE '%rent%' THEN {rating_points['good']}
    WHEN LOWER(c.home_ownership) LIKE '%mortgage%' THEN {rating_points['bad']}
    WHEN LOWER(c.home_ownership) LIKE '%any%' OR c.home_ownership IS NULL THEN {rating_points['verybad']}
END AS home_points,

-- Credit limits points
CASE 
    WHEN l.funded_amount > (c.total_high_credit_limit * 0.10) AND l.funded_amount <= (c.total_high_credit_limit * 0.20) THEN {rating_points['verygood']}
    WHEN l.funded_amount > (c.total_high_credit_limit * 0.20) AND l.funded_amount <= (c.total_high_credit_limit * 0.30) THEN {rating_points['good']}
    WHEN l.funded_amount > (c.total_high_credit_limit * 0.30) AND l.funded_amount <= (c.total_high_credit_limit * 0.50) THEN {rating_points['bad']}
    WHEN l.funded_amount > (c.total_high_credit_limit * 0.50) AND l.funded_amount <= (c.total_high_credit_limit * 0.70) THEN {rating_points['verybad']}
    WHEN l.funded_amount > (c.total_high_credit_limit * 0.70) THEN {rating_points['unacceptable']}
END AS cred_limits_points,

-- Grade points
CASE 
    WHEN c.grade = 'A' AND c.sub_grade = 'A1' THEN {rating_points['excellent']}
    WHEN c.grade = 'A' AND c.sub_grade = 'A2' THEN ({rating_points['excellent']} * 0.95)
    WHEN c.grade = 'A' AND c.sub_grade = 'A3' THEN ({rating_points['excellent']} * 0.9)
    WHEN c.grade = 'A' AND c.sub_grade = 'A4' THEN ({rating_points['excellent']} * 0.85)
    WHEN c.grade = 'A' AND c.sub_grade = 'A5' THEN ({rating_points['excellent']} * 0.8)

    WHEN c.grade = 'B' AND c.sub_grade = 'B1' THEN {rating_points['verygood']}
    WHEN c.grade = 'B' AND c.sub_grade = 'B2' THEN ({rating_points['verygood']} * 0.95)
    WHEN c.grade = 'B' AND c.sub_grade = 'B3' THEN ({rating_points['verygood']} * 0.9)
    WHEN c.grade = 'B' AND c.sub_grade = 'B4' THEN ({rating_points['verygood']} * 0.85)
    WHEN c.grade = 'B' AND c.sub_grade = 'B5' THEN ({rating_points['verygood']} * 0.8)

    WHEN c.grade = 'C' AND c.sub_grade = 'C1' THEN {rating_points['good']}
    WHEN c.grade = 'C' AND c.sub_grade = 'C2' THEN ({rating_points['good']} * 0.95)
    WHEN c.grade = 'C' AND c.sub_grade = 'C3' THEN ({rating_points['good']} * 0.9)
    WHEN c.grade = 'C' AND c.sub_grade = 'C4' THEN ({rating_points['good']} * 0.85)
    WHEN c.grade = 'C' AND c.sub_grade = 'C5' THEN ({rating_points['good']} * 0.8)

    WHEN c.grade = 'D' AND c.sub_grade = 'D1' THEN {rating_points['bad']}
    WHEN c.grade = 'D' AND c.sub_grade = 'D2' THEN ({rating_points['bad']} * 0.95)
    WHEN c.grade = 'D' AND c.sub_grade = 'D3' THEN ({rating_points['bad']} * 0.9)
    WHEN c.grade = 'D' AND c.sub_grade = 'D4' THEN ({rating_points['bad']} * 0.85)
    WHEN c.grade = 'D' AND c.sub_grade = 'D5' THEN ({rating_points['bad']} * 0.8)

    WHEN c.grade = 'E' AND c.sub_grade = 'E1' THEN {rating_points['verybad']}
    WHEN c.grade = 'E' AND c.sub_grade = 'E2' THEN ({rating_points['verybad']} * 0.95)
    WHEN c.grade = 'E' AND c.sub_grade = 'E3' THEN ({rating_points['verybad']} * 0.9)
    WHEN c.grade = 'E' AND c.sub_grade = 'E4' THEN ({rating_points['verybad']} * 0.85)
    WHEN c.grade = 'E' AND c.sub_grade = 'E5' THEN ({rating_points['verybad']} * 0.8)
END AS grade_points

FROM ph_dh_points p
JOIN silver_cleaned.loans l ON p.member_id = l.member_id
JOIN silver_enriched.customers c ON p.member_id = c.member_id
where p.member_id not in (select member_id from gold.bad_data_customer_final)
"""

ph_dh_fh_df = spark.sql(query)


In [0]:
ph_dh_fh_df.createOrReplaceTempView('ph_dh_fh_points')

In [0]:
#display(ph_dh_fh_df)

#Loan Score

In [0]:
query = """
SELECT *,
-- Payment history points
((last_payment_points + total_payment_points) * 0.2) AS payment_history_points,

-- Defaulter history points
((delinq_points + public_rec_points + public_bankrupticies_points + inquiry_points) * 0.45) AS defaulter_history,

-- Financial health points
((loan_status_points + home_points + cred_limits_points + grade_points) * 0.35) AS financial_health

FROM ph_dh_fh_points
"""

loan_score = spark.sql(query)


In [0]:
final_loan_score = loan_score.withColumn('loan_score',
                                        col('payment_history_points')+col('defaulter_history')+col('financial_health'))

In [0]:
final_loan_score.createOrReplaceTempView('loan_score_eval')

In [0]:
query = f"""
SELECT *,
-- Loan final grade assignment
CASE
    WHEN loan_score > {grade_points['verygood']} THEN 'A'
    WHEN loan_score <= {grade_points['verygood']} AND loan_score > {grade_points['good']} THEN 'B'
    WHEN loan_score <= {grade_points['good']} AND loan_score > {grade_points['bad']} THEN 'C'
    WHEN loan_score <= {grade_points['bad']} AND loan_score > {grade_points['verybad']} THEN 'D'
    WHEN loan_score <= {grade_points['verybad']} AND loan_score > {grade_points['unacceptable']} THEN 'E'
    WHEN loan_score <= {grade_points['unacceptable']} THEN 'F'
END AS loan_final_grade
FROM loan_score_eval
"""

load_score_grade = spark.sql(query)

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/final_loan_score/loan_score/", recurse=True)

In [0]:
load_score_grade.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/final_loan_score/loan_score/')

In [0]:
%sql
create or replace table final_loan_score
as
select * from delta.`/Volumes/lendingclub/storagelocation/final_loan_score/loan_score/`